In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
 
BASE        = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
ARQUIVO_IN  = BASE / "data/processed/pnadc_2023_anual.csv"
DIR_FIGS    = BASE / "outputs/figures"
DIR_FIGS.mkdir(parents=True, exist_ok=True)
 
plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})
 
print("Imports OK")

In [ ]:
df = pd.read_csv(ARQUIVO_IN, dtype={"VD4002": str, "V2007": str, "UF": str})
 
ocupados = df[df["VD4002"] == "1"].copy()
ocupados["VD4031"] = pd.to_numeric(ocupados["VD4031"], errors="coerce")
ocupados["VD4019"] = pd.to_numeric(ocupados["VD4019"], errors="coerce")
ocupados["V2009"]  = pd.to_numeric(ocupados["V2009"],  errors="coerce")
ocupados["V1028"]  = pd.to_numeric(ocupados["V1028"],  errors="coerce")
 
print(f"Ocupados: {len(ocupados):,}")
print(f"Com horas válidas: {ocupados['VD4031'].notna().sum():,}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
 
ax.hist(
    ocupados["VD4031"].dropna(),
    bins=range(1, 100, 2),
    color="#2C6FBF",
    edgecolor="white",
    linewidth=0.4,
    weights=[1 / len(ocupados["VD4031"].dropna())] * ocupados["VD4031"].notna().sum()
)
 
ax.axvline(40, color="#E84545", linewidth=1.8, linestyle="--", label="40h (proposta)")
ax.axvline(44, color="#F5A623", linewidth=1.8, linestyle="--", label="44h (atual CLT)")
 
ax.set_xlabel("Horas habituais semanais", fontsize=11)
ax.set_ylabel("Proporção de trabalhadores", fontsize=11)
ax.set_title(
    "Distribuição da jornada semanal habitual — Brasil 2023\n"
    "Trabalhadores ocupados · Fonte: PNAD Contínua, IBGE",
    fontsize=12, loc="left"
)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.legend(frameon=False, fontsize=10)
ax.set_xlim(0, 90)
 
plt.tight_layout()
plt.savefig(DIR_FIGS / "01_distribuicao_jornada_histograma.png", dpi=150)
plt.show()
print("Gráfico 1 salvo.")


In [ ]:
bins   = [0, 20, 30, 39, 40, 44, 48, 60, 120]
labels = ["até 20h", "21–30h", "31–39h", "40h exatas",
          "41–44h", "45–48h", "49–60h", "60h+"]
 
ocupados["faixa"] = pd.cut(ocupados["VD4031"], bins=bins, labels=labels, right=True)
 
dist     = ocupados["faixa"].value_counts().sort_index()
dist_pct = dist / dist.sum() * 100
 
cores = [
    "#B0C4DE", "#B0C4DE", "#B0C4DE", "#2C6FBF",
    "#F5A623", "#E84545", "#C0392B", "#8B0000"
]
 
fig, ax = plt.subplots(figsize=(7, 3))
 
bars = ax.barh(dist_pct.index, dist_pct.values, color=cores, edgecolor="white")
 
for bar, val in zip(bars, dist_pct.values):
    ax.text(
        bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}%", va="center", fontsize=10
    )
 
ax.set_xlabel("                                                                                      % dos trabalhadores ocupados", fontsize=10)
ax.set_title(
    "Faixas de jornada semanal habitual — Brasil 2023\n"
    "Trabalhadores ocupados · Fonte: PNAD Contínua, IBGE",
    fontsize=12, loc="left"
)
ax.set_xlim(0, 42)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))
 
ax.axvline(0, color="black", linewidth=0.5)
fig.text(
    0.13, 0.02,
    "Laranja = acima da proposta (40h) | Vermelho = acima da CLT atual (44h)",
    fontsize=9, color="gray"
)
 
plt.tight_layout()
plt.savefig(DIR_FIGS / "02_faixas_jornada_barras.png", dpi=150)
plt.show()
print("Gráfico 2 salvo.")

In [ ]:
setores = {
    "01": "Agropecuária",
    "02": "Indústria geral",
    "03": "Construção",
    "04": "Comércio e rep.",
    "05": "Transp. e armaz.",
    "06": "Alojamento e alim.",
    "07": "Informação e com.",
    "08": "Ativ. financeiras",
    "09": "Ativ. imobiliárias",
    "10": "Serv. profissionais",
    "11": "Adm. pública",
    "12": "Educação",
    "13": "Saúde",
    "14": "Outros serviços",
    "15": "Serv. domésticos",
}
 
ocupados["setor_nome"] = ocupados["VD4010"].map(setores)
 
media_setor = (
    ocupados.groupby("setor_nome")["VD4031"]
    .mean()
    .dropna()
    .sort_values(ascending=True)
)
 
fig, ax = plt.subplots(figsize=(5, 3))
 
cores_setor = [
    "#E84545" if v > 44 else "#F5A623" if v > 40 else "#2C6FBF"
    for v in media_setor.values
]
 
bars = ax.barh(media_setor.index, media_setor.values, color=cores_setor)
 
ax.axvline(40, color="#2C6FBF", linewidth=1.5, linestyle="--", alpha=0.7, label="40h")
ax.axvline(44, color="#F5A623", linewidth=1.5, linestyle="--", alpha=0.7, label="44h")
 
for bar, val in zip(bars, media_setor.values):
    ax.text(
        bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}h", va="center", fontsize=9
    )
 
ax.set_xlabel("Média de horas habituais semanais", fontsize=11)
ax.set_title(
    "Jornada média semanal por setor de atividade — Brasil 2023\n"
    "Trabalhadores ocupados · Fonte: PNAD Contínua, IBGE",
    fontsize=12, loc="left"
)
ax.set_xlim(0, 60)
ax.legend(frameon=False, fontsize=10)
 
plt.tight_layout()
plt.savefig(DIR_FIGS / "03_jornada_media_por_setor.png", dpi=150)
plt.show()
print("Gráfico 3 salvo.")

In [ ]:
print("\nArquivos gerados em outputs/figures/:")
for f in sorted(DIR_FIGS.glob("*.png")):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")
